In [54]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [55]:
# data = pd.read_stata(r"datos/ECU_2004m12_BID.dta", convert_categoricals=False) # para bases de stata

# data3 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2015/m3/data_orig/201503_EnemduBDD_15anios.sav", convert_categoricals=False) # para bases de stata
# data6 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2015/m6/data_orig/201506_EnemduBDD_15anios.sav", convert_categoricals=False) # para bases de stata
# data9 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2015/m9/data_orig/201509_EnemduBDD_15anios.sav", convert_categoricals=False) # para bases de stata
# data12 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2015/m12/data_orig/201512_EnemduBDD_15anios.sav", convert_categoricals=False) # para bases de stata

data3 = pd.read_spss(r"Z:\survey\ECU\ENEMDU\2015\m3\data_orig\201503_EnemduBDD_15anios.sav", convert_categoricals=False) # para bases de stata
data6 = pd.read_spss(r"Z:\survey\ECU\ENEMDU\2015\m6\data_orig\201506_EnemduBDD_15anios.sav", convert_categoricals=False) # para bases de stata
data9 = pd.read_spss(r"Z:\survey\ECU\ENEMDU\2015\m9\data_orig\201509_EnemduBDD_15anios.sav", convert_categoricals=False) # para bases de stata
data12 = pd.read_spss(r"Z:\survey\ECU\ENEMDU\2015\m12\data_orig\201512_EnemduBDD_15anios.sav", convert_categoricals=False) # para bases de stata

In [56]:
data3.columns = [x.lower() for x in data3.columns]
data6.columns = [x.lower() for x in data6.columns]
data9.columns = [x.lower() for x in data9.columns]
data12.columns = [x.lower() for x in data12.columns]

In [57]:
data12[['area', 'ciudad', 'zona', 'sector', 'panelm', 'vivienda', 'hogar',
       'p02', 'p03', 'p66', 'fexp', 'p20']]

,area,ciudad,zona,sector,panelm,vivienda,hogar,p02,p03,p66,fexp,p20
0,1.0,10150.0,009,004,22.0,1.0,1.0,1.0,26.0,NaN,141.065750,1.0
1,1.0,10150.0,056,005,12.0,3.0,1.0,1.0,68.0,NaN,65.976281,2.0
2,1.0,70150.0,005,001,22.0,2.0,2.0,1.0,41.0,560.0,123.512430,1.0
3,1.0,70150.0,043,004,22.0,1.0,1.0,1.0,38.0,1200.0,96.697327,1.0
4,1.0,80150.0,012,003,12.0,1.0,1.0,1.0,64.0,NaN,89.319663,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
112816,2.0,900451.0,999,007,122.0,2.0,1.0,2.0,6.0,NaN,131.482886,2.0
112817,2.0,900451.0,999,007,122.0,2.0,1.0,2.0,33.0,NaN,131.482886,2.0
112818,2.0,900451.0,999,007,122.0,2.0,1.0,1.0,33.0,320.0,131.482886,1.0
112819,2.0,900451.0,999,007,122.0,3.0,1.0,1.0,88.0,NaN,131.482886,2.0


## Revisar los datos

| marzo | junio | septiembre | diciembre |
|-----------|-----------|-----------|-----------|
| area  | area  | area  | area  |
| ciudad  | ciudad  | ciudad  | ciudad  |
| zona  | zona  | zona  | zona  |
| sector  | sector  | sector  | sector  |
| panelm  | panelm  | panelm  | panelm  |
| vivienda  | vivienda  | vivienda  | vivienda  |
| hogar  | hogar  | hogar  | hogar  |
| p02  | p02  | p02  | p02  |
| p03  | p03  | p03  | p03  |
| p66  | p66  | p66  | p66  |
| fexp  | fexp  | fexp  | fexp  |
| p20  | p20  | p20  | p20  |

En esta encuesta tenemos separadas cuatro diferentes bases para cada trimestre, esto cambia la lógica que habíamos tenido hasta ahora así que de aquí en adelante cambiamos algo del código, mantenemos de acuerdo a las etiquetas de las variables pe63 como la variable de ingreso laboral monetario de la actividad principal asalariada para mantener la concordancia con el resto de los años.

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no trabajando, estas variables las usamos antes para identificar la condición de trabajo para diferentes meses en encuestas anuales o incompletas donde asumíamos que mantenía el mismo salario si estaba ocupado en ese mes, sin mbargo estas variables tenían el problema de no corresponder de forma exacta con el año o mes de la encuesta. Ahora sin embargo podemos cambiar las suposiciones y solamente asumir que si la variable 'trabajando' que pregunta si el individuo trabajó la semana pasada se cumple vamos a asumir que trabajo durante todo el trimestre, de esta manera podemos mejorar las suposiciones de ocupación mensual, mantenemos la idea de que si el individuo trabajo recibe su ingreso laboral reportado.

In [58]:
columnas = pd.Index(['area', 'ciudad', 'zona', 'sector', 'panelm',
            'vivienda', 'hogar', 'p66',
            'fexp', 'p02', 'p03', 'p20'])

Filtramos solo las columnas de interés para alivar el peso en la memoria

In [59]:
data3 = data3[columnas.intersection(data3.columns)]
data6 = data6[columnas.intersection(data6.columns)]
data9 = data9[columnas.intersection(data9.columns)]
data12 = data12[columnas.intersection(data12.columns)]

Creamos una variable de ingreso laboral que es igual al ingreso por asalariado y limpiamos según los valores de ingrl, para mantener ambas variables para cada base consistente

In [60]:
data12['p66'].value_counts().sort_index(ascending=False)

p66
999999.0     114
14000.0        1
6500.0         1
5500.0         1
5200.0         1
            ... 
8.0            2
5.0            2
4.0            1
1.0            1
0.0         1569
Name: count, Length: 960, dtype: int64

In [61]:
data3['p66'] = pd.to_numeric(data3['p66'], errors='coerce')
data3['p66'] = data3['p66'].apply(lambda x: np.nan if x > 8000 else x)
data3['p66'] = data3['p66'].apply(lambda x: np.nan if x < 0 else x)

data6['p66'] = pd.to_numeric(data6['p66'], errors='coerce')
data6['p66'] = data6['p66'].apply(lambda x: np.nan if x > 9552 else x)
data6['p66'] = data6['p66'].apply(lambda x: np.nan if x < 0 else x)

data9['p66'] = pd.to_numeric(data9['p66'], errors='coerce')
data9['p66'] = data9['p66'].apply(lambda x: np.nan if x > 8000 else x)
data9['p66'] = data9['p66'].apply(lambda x: np.nan if x < 0 else x)

data12['p66'] = pd.to_numeric(data12['p66'], errors='coerce')
data12['p66'] = data12['p66'].apply(lambda x: np.nan if x > 14000 else x)
data12['p66'] = data12['p66'].apply(lambda x: np.nan if x < 0 else x)

In [62]:
data3['ingr'] = data3['p66']
data6['ingr'] = data6['p66']
data9['ingr'] = data9['p66']
data12['ingr'] = data12['p66']

Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingr' siempre que reportan estar ocupados la semana pasada, de acuerdo a la variable 'trabajo'

In [63]:
data3['ingr_t1'] = data3.apply(lambda x: x['ingr'] if x['p20'] == 1 else np.nan, axis=1)

data6['ingr_t2'] = data6.apply(lambda x: x['ingr'] if x['p20'] == 1 else np.nan, axis=1)

data9['ingr_t3'] = data9.apply(lambda x: x['ingr'] if x['p20'] == 1 else np.nan, axis=1)

data12['ingr_t4'] = data12.apply(lambda x: x['ingr'] if x['p20'] == 1 else np.nan, axis=1)

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014

In [64]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 2015]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc

In [65]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Sierra': fila['Sierra'],
        'Costa': fila['Costa'],
        'Guayaquil': fila['Guayaquil'],
        'Esmeraldas': fila['Esmeraldas'],
        'Machala': fila['Machala'],
        'Manta': fila['Manta'],
        'Quito': fila['Quito'],
        'Loja': fila['Loja'],
        'Cuenca': fila['Cuenca'],
        'Ambato': fila['Ambato']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Sierra': fila['Sierra'],
        'Costa': fila['Costa'],
        'Guayaquil': fila['Guayaquil'],
        'Esmeraldas': fila['Esmeraldas'],
        'Machala': fila['Machala'],
        'Manta': fila['Manta'],
        'Quito': fila['Quito'],
        'Loja': fila['Loja'],
        'Cuenca': fila['Cuenca'],
        'Ambato': fila['Ambato']
    }
     for _, fila in datos_base.iterrows()
     }

In [66]:
data3['ciudad'][0]

10150.0

### Creamos identificadores para las ciudades siguiendo los códigos del INEC y para los trimestres

In [67]:
# Corregimos los códigos para usarlos cómo texto
data3['ciudad'] = data3['ciudad'].apply(str)
data3['ciudad'] = data3['ciudad'].apply(lambda x: '0' + x if len(x) == 7 else x)
data3['ciudad_2'] = data3['ciudad'].apply(lambda x: x[:4])

data6['ciudad'] = data6['ciudad'].apply(str)
data6['ciudad'] = data6['ciudad'].apply(lambda x: '0' + x if len(x) == 7 else x)
data6['ciudad_2'] = data6['ciudad'].apply(lambda x: x[:4])

data9['ciudad'] = data9['ciudad'].apply(str)
data9['ciudad'] = data9['ciudad'].apply(lambda x: '0' + x if len(x) == 7 else x)
data9['ciudad_2'] = data9['ciudad'].apply(lambda x: x[:4])

data12['ciudad'] = data12['ciudad'].apply(str)
data12['ciudad'] = data12['ciudad'].apply(lambda x: '0' + x if len(x) == 7 else x)
data12['ciudad_2'] = data12['ciudad'].apply(lambda x: x[:4])

Diccionario ciudades disponibles

In [68]:
parroquia_dict = {
    '0101': 'Cuenca',
    '0901': 'Guayaquil',
    '0801': 'Esmeraldas',
    '0701': 'Machala',
    '1308': 'Manta',
    '1701': 'Quito',
    '1101': 'Loja',
    '1801': 'Ambato'
}

def get_parroquia(codigo):
    if codigo in parroquia_dict:
        return parroquia_dict[codigo]
    elif codigo[:2] in ['01', '02', '03', '04', '05', '06', '10', '11', '17', '18']:
        return 'Sierra'
    elif codigo[:2] in ['07', '08', '09', '12', '13', '23', '24']:
        return 'Costa'
    else:
        return 'Nacional'

data3['ciudad_asignada'] = data3['ciudad_2'].apply(get_parroquia)
data6['ciudad_asignada'] = data6['ciudad_2'].apply(get_parroquia)
data9['ciudad_asignada'] = data9['ciudad_2'].apply(get_parroquia)
data12['ciudad_asignada'] = data12['ciudad_2'].apply(get_parroquia)

In [69]:
data3['ciudad_asignada'].value_counts()

ciudad_asignada
Sierra        13468
Costa         12325
Nacional      11179
Guayaquil      5576
Quito          4635
Ambato         4104
Cuenca         3610
Machala        3008
Esmeraldas     1057
Loja            810
Manta           493
Name: count, dtype: int64

### Asignamos el ipc correspondiente según ciudad correspondiente

$\begin{equation}
    ingr_{USD-base-2014}^{i} = ingr_{dólares}^{i}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Desde 2000 en adelante ya no es necesario utilizar el tipo de cambio debido al cambio de moneda

In [70]:
# Función que asigna valores correspondientes
def asigna_ipc(fila, trimestre):
    return ipc_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila, trimestre):
    return ipc_base_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

In [71]:
data3['ipc_t1'] = data3.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
data3['ipc_base_t1'] = data3.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)

data6['ipc_t2'] = data6.apply(lambda fila: asigna_ipc(fila, 2), axis=1)
data6['ipc_base_t2'] = data6.apply(lambda fila: asigna_ipc_base(fila, 2), axis=1)

data9['ipc_t3'] = data9.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data9['ipc_base_t3'] = data9.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)

data12['ipc_t4'] = data12.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data12['ipc_base_t4'] = data12.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)

In [72]:
# Calculamos el deflactor
data3['def_t1'] = (data3['ipc_base_t1'] / data3['ipc_t1'])
data6['def_t2'] = (data6['ipc_base_t2'] / data6['ipc_t2'])
data9['def_t3'] = (data9['ipc_base_t3'] / data9['ipc_t3'])
data12['def_t4'] = (data12['ipc_base_t4'] / data12['ipc_t4'])

Ingreso promedio en el trimeste

In [73]:
data3['ingr_t1_r'] = data3['ingr_t1'] * data3['def_t1']
data6['ingr_t2_r'] = data6['ingr_t2'] * data6['def_t2']
data9['ingr_t3_r'] = data9['ingr_t3'] * data9['def_t3']
data12['ingr_t4_r'] = data12['ingr_t4'] * data12['def_t4']

In [74]:
print(data3['ingr_t1_r'].mean())
print(data6['ingr_t2_r'].mean())
print(data9['ingr_t3_r'].mean())
print(data12['ingr_t4_r'].mean())

443.7001540658992
439.80715037418315
523.6499245886502
441.66498630898985


## Regiones

In [75]:
# Corregimos los códigos para usarlos cómo texto
data3['ciudad'] = data3['ciudad'].apply(str)
data3['ciudad'] = data3['ciudad'].apply(lambda x: '0' + x if len(x) == 7 else x)

data3['ciudad_2'] = data3['ciudad'].apply(lambda x: x[:2])

data6['ciudad'] = data6['ciudad'].apply(str)
data6['ciudad'] = data6['ciudad'].apply(lambda x: '0' + x if len(x) == 7 else x)

data6['ciudad_2'] = data6['ciudad'].apply(lambda x: x[:2])

data9['ciudad'] = data9['ciudad'].apply(str)
data9['ciudad'] = data9['ciudad'].apply(lambda x: '0' + x if len(x) == 7 else x)

data9['ciudad_2'] = data9['ciudad'].apply(lambda x: x[:2])

data12['ciudad'] = data12['ciudad'].apply(str)
data12['ciudad'] = data12['ciudad'].apply(lambda x: '0' + x if len(x) == 7 else x)

data12['ciudad_2'] = data12['ciudad'].apply(lambda x: x[:2])

In [76]:
regiones_dict = {
    'Guayas': '09',
    'Manabí': '13',
    'El Oro': '07',
    'Los Ríos': '12',
    'Pichincha': '17',
    'Azuay': '01',
    'Galápagos': '20',
    'Sierra': ['04', '10', '05', '18', '02', '06', '03', '11'],
    'Costa, Santo Domingo': ['08', '24', '23'],
    'Amazonía': ['14', '15', '16', '19', '21', '22', '90']
}

In [77]:
codigo_region = {}
for region, codes in regiones_dict.items():
    
    if isinstance(codes, list):
        for code in codes:
            codigo_region[code] = region
    
    else:
        codigo_region[codes] = region

# Mapeo de regiones
data3['region'] = data3['ciudad_2'].map(codigo_region)

data6['region'] = data6['ciudad_2'].map(codigo_region)

data9['region'] = data9['ciudad_2'].map(codigo_region)

data12['region'] = data12['ciudad_2'].map(codigo_region)

In [78]:
data3['region'].value_counts()

region
Sierra                  17203
Amazonía                11179
Costa, Santo Domingo     6561
Guayas                   6301
Pichincha                5243
El Oro                   4435
Azuay                    4181
Manabí                   2590
Los Ríos                 2572
Name: count, dtype: int64

In [79]:
data6['region'].value_counts()

region
Sierra                  38908
Amazonía                23237
Costa, Santo Domingo    13694
Guayas                   8253
Pichincha                7275
El Oro                   6369
Azuay                    5509
Manabí                   5347
Los Ríos                 5338
Galápagos                1059
Name: count, dtype: int64

In [80]:
data9['region'].value_counts()

region
Sierra                  16726
Amazonía                10882
Guayas                   6239
Costa, Santo Domingo     6197
Pichincha                4976
El Oro                   4297
Azuay                    4059
Los Ríos                 2564
Manabí                   2504
Name: count, dtype: int64

In [81]:
data12['region'].value_counts()

region
Sierra                  38382
Amazonía                22351
Costa, Santo Domingo    13465
Guayas                   8250
Pichincha                7040
El Oro                   6131
Azuay                    5529
Manabí                   5389
Los Ríos                 5306
Galápagos                 978
Name: count, dtype: int64

## Calculo ingreso de los hogares

In [82]:
columnas_idef = pd.Index(['area', 'ciudad', 'zona', 'sector', 'vivienda',
       'hogar'])

data3['idef_hogar'] = data3[columnas_idef].astype(str).agg(''.join, axis=1)
data6['idef_hogar'] = data6[columnas_idef].astype(str).agg(''.join, axis=1)
data9['idef_hogar'] = data9[columnas_idef].astype(str).agg(''.join, axis=1)
data12['idef_hogar'] = data12[columnas_idef].astype(str).agg(''.join, axis=1)

print(len(data3['idef_hogar'].unique()))
print(len(data6['idef_hogar'].unique()))
print(len(data9['idef_hogar'].unique()))
print(len(data12['idef_hogar'].unique()))

4637
9257
4919
9655


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [83]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [84]:
data3['ingr_t1_h'] = data3.groupby('idef_hogar')['ingr_t1_r'].transform(sum_with_na)
data6['ingr_t2_h'] = data6.groupby('idef_hogar')['ingr_t2_r'].transform(sum_with_na)
data9['ingr_t3_h'] = data9.groupby('idef_hogar')['ingr_t3_r'].transform(sum_with_na)
data12['ingr_t4_h'] = data12.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)

In [85]:
print(data3['ingr_t1_h'].mean())
print(data6['ingr_t2_h'].mean())
print(data9['ingr_t3_h'].mean())
print(data12['ingr_t4_h'].mean())

1478.2276532106087
1451.8825520704834
1691.6561484640831
1396.220848190514


## Sacamos edades negativas y mayores a 100 años

In [86]:
print(len(data3))
print(len(data6))
print(len(data9))
print(len(data12))

60265
114989
58444
112821


Transformamos las variables de edad a numericas para evitar problemas

In [87]:
data3['edad'] = pd.to_numeric(data3['p03'], errors='coerce')
data6['edad'] = pd.to_numeric(data6['p03'], errors='coerce')
data9['edad'] = pd.to_numeric(data9['p03'], errors='coerce')
data12['edad'] = pd.to_numeric(data12['p03'], errors='coerce')

In [88]:
data3 = data3.loc[(data3['edad'] >= 0) & (data3['edad'] < 100)]
data6 = data6.loc[(data6['edad'] >= 0) & (data6['edad'] < 100)]
data9 = data9.loc[(data9['edad'] >= 0) & (data9['edad'] < 100)]
data12 = data12.loc[(data12['edad'] >= 0) & (data12['edad'] < 100)]

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [89]:
k = 0.4
s = 0.9

In [90]:
# Si es necesario calcular el número de niños
data3['es_nino'] = data3['edad'] < 10
data3['ninos'] = data3.groupby('idef_hogar')['es_nino'].transform('sum')

data6['es_nino'] = data6['edad'] < 10
data6['ninos'] = data6.groupby('idef_hogar')['es_nino'].transform('sum')

data9['es_nino'] = data9['edad'] < 10
data9['ninos'] = data9.groupby('idef_hogar')['es_nino'].transform('sum')

data12['es_nino'] = data12['edad'] < 10
data12['ninos'] = data12.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data3['es_adulto'] = data3['edad'] > 10
data3['adultos'] = data3.groupby('idef_hogar')['es_adulto'].transform('sum')

data6['es_adulto'] = data6['edad'] > 10
data6['adultos'] = data6.groupby('idef_hogar')['es_adulto'].transform('sum')

data9['es_adulto'] = data9['edad'] > 10
data9['adultos'] = data9.groupby('idef_hogar')['es_adulto'].transform('sum')

data12['es_adulto'] = data12['edad'] > 10
data12['adultos'] = data12.groupby('idef_hogar')['es_adulto'].transform('sum')

In [91]:
data3['escala'] = (data3['adultos'] + k * data3['ninos']) ** s
data6['escala'] = (data6['adultos'] + k * data6['ninos']) ** s
data9['escala'] = (data9['adultos'] + k * data9['ninos']) ** s
data12['escala'] = (data12['adultos'] + k * data12['ninos']) ** s

In [92]:
data3['ingr_t_t1'] = data3['ingr_t1_h'] / data3['escala']
data6['ingr_t_t2'] = data6['ingr_t2_h'] / data6['escala']
data9['ingr_t_t3'] = data9['ingr_t3_h'] / data9['escala']
data12['ingr_t_t4'] = data12['ingr_t4_h'] / data12['escala']

In [93]:
print(data3['ingr_t_t1'].mean())
print(data6['ingr_t_t2'].mean())
print(data9['ingr_t_t3'].mean())
print(data12['ingr_t_t4'].mean())

150.67760536312483
153.362444202917
185.42528736900226
154.70632755632383


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [94]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))
salario_dict = dict(zip(datos_actual['trimestre'], datos_actual['salario básico unificado']))
ano = 2015

In [95]:
umbral_dict

{1: 77.8823883305729,
 2: 78.2776170788467,
 3: 79.8829584681759,
 4: 80.6756338864045}

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [96]:
resultados_list = []

# Para cada trimeste
for t in [1, 2, 3, 4]:
    col_ingr = f'ingr_t_t{t}'
    umbral = umbral_dict.get(t)
    salario = salario_dict.get(t)
    
    # Selecciona el dataframe correspondiente
    if t == 1:
        df_actual = data3
    elif t == 2:
        df_actual = data6
    elif t == 3:
        df_actual = data9
    elif t == 4:
        df_actual = data12
    else:
        continue

    # Agrupa por región
    grouped = df_actual.groupby('region')
    
    for region_name, group in grouped:
        # 1. Filtra datos
        valid = group.dropna(subset=[col_ingr])
        
        if len(valid) == 0:
            continue
            
        # Extrae los vectores 
        ingresos = valid[col_ingr].values
        pesos = valid['fexp'].values
        
        # 2. Calcula indices
        gaps = (umbral - ingresos) / umbral
        gaps = np.clip(gaps, a_min=0, a_max=None)
        
        # 3. Calcula FGT
        total_poblacion = pesos.sum()
        
        # FGT0
        fgt0 = (pesos * (gaps > 0).astype(int)).sum() / total_poblacion
        
        # FGT1
        fgt1 = (pesos * (gaps ** 1)).sum() / total_poblacion
        
        # FGT2
        fgt2 = (pesos * (gaps ** 2)).sum() / total_poblacion
        
        # 4. Calcula Ingreso promedio
        ingreso_promedio = np.average(ingresos, weights=pesos)

        # 5. Desigualdad de Atkinson
        atkinson_resultados = {}
        
        if ingreso_promedio > 0:
            for epsilon in [0.25, 0.5, 0.75]:
                # La suma ponderada de la utilidad
                utility_sum = np.sum((ingresos ** (1 - epsilon)) * pesos)
                
                # promedio de esa utilidad
                utility_mean = utility_sum / total_poblacion
                
                # ingreso equivalente
                y_ede = utility_mean ** (1 / (1 - epsilon))
                
                # índice final
                atkinson_index = 1 - (y_ede / ingreso_promedio)
                atkinson_resultados[f'a{int(epsilon*100)}'] = atkinson_index
        else:
            # Si nadie gana nada, definimos desigualdad como NaN
            atkinson_resultados = {'a25': np.nan, 'a50': np.nan, 'a75': np.nan}

        # 6. Calcula mediana del ingreso
        # Ordena
        sort_idx = np.argsort(ingresos)
        ingreso_ordenado = ingresos[sort_idx]
        pesos_ordenado = pesos[sort_idx]
        cumsum_pesos = np.cumsum(pesos_ordenado)
        cutoff = total_poblacion / 2.0
        mediana = ingreso_ordenado[np.searchsorted(cumsum_pesos, cutoff)]
        
        # 7. Guarda resultados
        resultados_list.append({
            'ano': ano,
            'trimestre': t,
            'region': region_name,
            'fgt0': fgt0,
            'fgt1': fgt1,
            'fgt2': fgt2,
            'a25': atkinson_resultados['a25'],
            'a50': atkinson_resultados['a50'],
            'a75': atkinson_resultados['a75'],
            'ingreso_promedio': ingreso_promedio,
            'ingreso_mediana': mediana + 1 if mediana < 1 else mediana,
            'salario_minimo': salario,
            'kaitz_indice': salario / (mediana + 1 if mediana < 1 else mediana)         
        })

# lista a DataFrame
df_final_regional = pd.DataFrame(resultados_list)
df_final_regional

,ano,trimestre,region,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio,ingreso_mediana,salario_minimo,kaitz_indice
0,2015,1,Amazonía,0.391295,0.195694,0.123511,0.081725,0.164533,0.249812,131.656671,106.137267,354.0,3.335304
1,2015,1,Azuay,0.272504,0.070591,0.026678,0.054930,0.107669,0.158618,156.228554,125.570411,354.0,2.819135
2,2015,1,"Costa, Santo Domingo",0.404155,0.163998,0.094529,0.063583,0.127565,0.199974,113.438923,97.027127,354.0,3.648464
3,2015,1,El Oro,0.222315,0.059043,0.026261,0.038180,0.075535,0.112292,145.461766,131.399031,354.0,2.694084
4,2015,1,Guayas,0.248232,0.088549,0.045460,0.052356,0.104641,0.159961,153.151232,135.085060,354.0,2.620571
5,2015,1,Los Ríos,0.351700,0.104859,0.047609,0.057038,0.106983,0.151622,120.230964,99.886323,354.0,3.544029
6,2015,1,Manabí,0.466849,0.155283,0.074755,0.053325,0.104278,0.153162,106.558714,83.381644,354.0,4.245539
7,2015,1,Pichincha,0.095529,0.034088,0.019601,0.055305,0.109893,0.170432,230.184905,193.340064,354.0,1.830971
8,2015,1,Sierra,0.367488,0.160051,0.093295,0.069764,0.139708,0.213966,127.817595,99.585701,354.0,3.554727
9,2015,2,Amazonía,0.333345,0.150281,0.088281,0.070862,0.142813,0.219200,144.075436,118.223204,354.0,2.994336


### Inserta los cálculos en la base final

In [97]:
indices = pd.read_csv("indices_region.csv", encoding='latin-1')

In [98]:
import os

# 2. cheque el archivo
if not os.path.isfile('indices_region.csv'):
    # Headers si es la primera vez
    df_final_regional.to_csv('indices_region.csv', index=False, encoding='latin-1')
else:
    # SI ya existe, append
    df_final_regional.to_csv('indices_region.csv', mode='a', index=False, header=False, encoding='latin-1')